In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Utilisation des workloads

<span id="usage"></span>
L'utilisation est une mesure du temps pendant lequel le QPU est verrouillé pour ton workload. Son calcul varie selon le mode d'exécution utilisé.

* L'utilisation d'une session correspond au temps réel (wall-clock) pendant lequel la session est active. Voir [Durée de session](/guides/run-jobs-session#session-length) pour plus d'informations sur les transitions d'état de session.
* L'utilisation d'un batch correspond à la somme du temps quantique (temps passé par le complexe QPU à traiter ton job) de tous les jobs du batch.
* L'utilisation d'un job individuel correspond au temps quantique utilisé par le job lors de son traitement.

Note que les jobs échoués ou annulés sont comptabilisés dans ton utilisation dans certaines circonstances — consulte la section [Jobs échoués et annulés](#failed-job) pour plus de détails.

Pour les utilisateurs d'un plan payant, l'utilisation détermine le coût du workload. Voir [Gérer les coûts](/guides/manage-cost) pour plus de détails.

<span id="failed-job"></span>
## Utilisation pour les jobs échoués et annulés
Quand un job échoue ou est annulé, l'utilisation rapportée est la suivante :

* Mode job ou batch : L'utilisation rapportée correspond au temps pendant lequel le QPU était verrouillé pour l'exécution de ton workload jusqu'au moment de l'échec ou de l'annulation. Par conséquent, si l'échec ou l'annulation s'est produit avant le verrouillage, l'utilisation rapportée est nulle. Sinon, l'utilisation rapportée du workload correspond au montant d'utilisation accumulé avant l'échec ou l'annulation. Ainsi, certains jobs échoués n'apparaissent pas dans ton utilisation rapportée, tandis que d'autres oui.

* Mode session : L'utilisation rapportée correspond au temps réel (wall-clock) pendant lequel la session est active, indépendamment du nombre de jobs qui échouent ou sont annulés.

<span id="view-usage"></span>
## Consulter l'utilisation réelle d'un workload
Une fois un workload terminé, plusieurs façons permettent de consulter son utilisation réelle :

- Lance [`batch.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/batch#usage) ou [`session.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session#usage) dans `qiskit-ibm-runtime` version 0.30 ou ultérieure. Si tu utilises une version plus ancienne de `qiskit-ibm-runtime` (>= 0.23 et < 0.30), l'utilisation peut quand même être trouvée dans `session.details()["usage_time"]` et `batch.details()["usage_time"]`.
- Utilise [`GET /sessions/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/sessions#tags__sessions__operations__GetSessionDetailsExtendedController_getSessionDetails) pour voir l'utilisation d'un batch ou d'une session spécifique.
- Utilise [`GET /jobs/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/jobs#tags__jobs__operations__GetJobByIdController_getJobById) pour voir l'utilisation d'un job individuel.

<span id="instance-usage"></span>
## Voir l'utilisation d'une instance
Tu peux consulter l'utilisation d'une instance sur la page [Instances](https://quantum.cloud.ibm.com/instances) ou, pour ceux disposant des autorisations appropriées, sur la page [Analytics](https://quantum.cloud.ibm.com/analytics). Note que ces pages peuvent afficher des chiffres d'utilisation différents car elles calculent l'utilisation de manière distincte.

La page Instances affiche l'utilisation en temps réel pour les 28 derniers jours (glissants), jusqu'à l'heure actuelle du jour en cours. L'utilisation affichée sur la page Analytics est recalculée toutes les heures et inclut les 28 derniers jours complets ; autrement dit, elle affiche l'utilisation depuis 00:00 il y a 28 jours jusqu'à aujourd'hui, à l'heure ronde précédente.

## Estimer l'utilisation avant de soumettre un job
Bien qu'obtenir une estimation locale précise soit compliqué par les opérations supplémentaires effectuées pour la suppression et l'atténuation des erreurs, tu peux utiliser cette formule de base pour obtenir une approximation de l'utilisation estimée :

`<per sub-job overhead> + (rep_delay + <circuit length>) * <num executions>`

- `<per sub-job overhead>` est une surcharge d'environ 2s par sous-job. Cela inclut des opérations telles que le chargement du payload dans l'électronique de contrôle. Ton job de primitive peut être divisé en plusieurs sous-jobs s'il est trop volumineux pour que le moteur d'exécution le traite en une seule fois.
- `rep_delay` est une option [personnalisable par l'utilisateur](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2#rep_delay), et sa valeur par défaut est donnée par `backend.default_rep_delay`, qui est de 250 microsecondes sur la plupart des backends IBM Quantum. Note que réduire `rep_delay` diminue le temps total d'exécution sur le QPU, mais au détriment d'un taux d'erreur de préparation d'état plus élevé ; consulte le guide [Exécution à taux de répétition dynamique](/guides/repetition-rate-execution) pour plus d'informations.
- `<circuit length>` est la longueur totale des instructions. Chaque instruction prend un temps différent sur le QPU, donc la longueur totale varie d'un circuit à l'autre. Une mesure, par exemple, peut prendre 56 fois plus de temps qu'une porte `x`. `backend.target[<instruction>][<qubit>].duration` peut être utilisé pour trouver la durée exacte de chaque instruction. Une longueur de circuit typique se situe généralement entre 50 et 100 microsecondes. Si tu utilises des techniques de suppression ou d'atténuation des erreurs avec les primitives, des instructions supplémentaires pourraient être insérées dans ton circuit, ce qui augmenterait la longueur totale du circuit.
    > **Note:** L'[option expérimentale `scheduler_timing`](/guides/visualize-circuit-timing) retourne le temps total du circuit, mais ce n'est PAS le temps utilisé pour la facturation.
- `<num executions>` est le nombre total de circuits multiplié par le nombre de shots, où les circuits sont ceux générés après la diffusion des éléments PUB.
    - Si tu utilises des techniques d'atténuation des erreurs avec les primitives, des circuits supplémentaires pourraient être exécutés dans le cadre du processus d'atténuation, ce qui augmenterait le nombre total d'exécutions. De plus, les techniques d'atténuation des erreurs avancées telles que PEA et PEC entraînent une surcharge bien plus importante car elles nécessitent l'exécution de circuits pour l'apprentissage du bruit.
    - L'Estimator regroupe les observables commutant qubit par qubit, ce qui réduit le nombre d'exécutions.

Si tu n'utilises pas de techniques d'atténuation des erreurs avancées ni de `rep_delay` personnalisé, tu peux utiliser `2+0.00035*<num executions>` comme formule rapide.

## Étapes suivantes
> **Tip:** - Consulte ces conseils : [Minimiser le temps d'exécution des jobs](minimize-time).
>     - Définis le [Temps d'exécution maximum](max-execution-time).
>     - Apprends à transpiler localement dans la section [Transpiler](./transpile/).
>     - Essaie le guide [Comparer les paramètres du transpileur](/guides/circuit-transpilation-settings).